# Libs & Setup

In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import logging
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import talib as ta
from torch import optim
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split
from tqdm import tqdm
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.optim import AdamW
from utils import scale, inverse_scale, inverse_scale_pair, inspect
from utils.paths import CHECKPOINTS_DIR, REPORTS_SIM_DIR, RESULTS_DIR
from pypfopt import risk_models, expected_returns, plotting, EfficientFrontier

# Own Libs
from config import *
from entities import *
from strategies import *
from datasets import *
from engine import Engine
from models import DiffusionTransformer, Diffusion

/home/narodom.y@FUSION.LAB/.conda/envs/tsgenai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# logging.basicConfig(level=logging.DEBUG)
logging.getLogger('matplotlib').setLevel(logging.WARNING)

In [3]:
cfg = TrainConfig(epochs=2, window_size=64, device=torch.device("cuda:1"))
window_size = cfg.window_size
device = cfg.device
batch_size = cfg.batch_size
epochs = cfg.epochs
sim_steps = cfg.steps_to_sim
num_sims = cfg.num_sims

# Optimizer
weight_decay = cfg.optimizer.weight_decay
lr = cfg.optimizer.lr
stride = 5
time_range = {
    'start_date': '2021-01-01',
    'end_date': '2024-12-31'
}

ddpm = {
    'timesteps': int(1000),
    'beta_start': 0.0001,
    'beta_end': 0.02
}

ddpm_transformer = {
    'window_size': window_size,
    'd_model': 64,
    'nhead': 4,
    'num_layers': 32,
    'dim_feedforward': 512,
    'dropout': 0.1
}

ckpt_name = f"ddpm_transformer_d{ddpm_transformer['d_model']}_l{ddpm_transformer['num_layers']}"

In [4]:
def time_range_info(df):
    info = (df.index.min(), df.index.max())
    print(f"Data range: {info[0]} to {info[1]}")
    
    duration = df.index.max() - df.index.min()
    print(f"Total duration: {duration}")

def time_range_mask(df, start_date, end_date):
    mask = (df.index >= start_date) & (df.index <= end_date)
    return mask

# Data

In [5]:
symbols = ['AAPL', 'TSLA', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'GOOG', 'META', 'AVGO', 'ORCL', 'CRM', 'ADBE', 'AMD', 'CSCO']
freq = "1d"

# Basket
basket = Basket(symbols=symbols)
basket.load_all_assets(freq=freq)

print(f"Basket data shape: {basket.data.shape}")
basket.data.head(5)

Basket data shape: (2760, 70)


AAPL                                                \
                Close       High        Low       Open       Volume   
Date                                                                  
2015-01-02  24.261047  24.729270  23.821672  24.718174  212818400.0   
2015-01-05  23.577574  24.110150  23.391173  24.030263  257142000.0   
2015-01-06  23.579794  23.839424  23.218085  23.641928  263188400.0   
2015-01-07  23.910433  24.010290  23.677430  23.788384  160423600.0   
2015-01-08  24.829119  24.886815  24.121236  24.238848  237458000.0   

                 TSLA                                             ...   AMD  \
                Close       High        Low       Open    Volume  ... Close   
Date                                                              ...         
2015-01-02  14.620667  14.883333  14.217333  14.858000  71466000  ...  2.67   
2015-01-05  14.006000  14.433333  13.810667  14.303333  80527500  ...  2.66   
2015-01-06  14.085333  14.280000  13.614000  14.004000  93928500  ...  2.63   
2015-01-07  14.063333  14.318667  13.985333  14.223333  44526000  ...  2.58   
2015-01-08  14.041333  14.253333  14.000667  14.187333  51637500  ...  2.61   

                                               CSCO                        \
            High   Low  Open      Volume      Close       High        Low   
Date                                                                        
2015-01-02  2.67  2.67  2.67         0.0  19.815605  20.181631  19.650534   
2015-01-05  2.70  2.64  2.67   8878200.0  19.420874  19.700776  19.377812   
2015-01-06  2.66  2.55  2.65  13912500.0  19.413698  19.865848  19.406522   
2015-01-07  2.65  2.54  2.63  12377600.0  19.593126  19.664896  19.363463   
2015-01-08  2.65  2.56  2.59  11136600.0  19.743843  20.160107  19.715135   

                                   
                 Open      Volume  
Date                               
2015-01-02  19.995029  22926500.0  
2015-01-05  19.607475  29460600.0  
2015-01-06  19.478291  47297600.0  
2015-01-07  19.478295  27570800.0  
2015-01-08  19.765374  40907000.0  

[5 rows x 70 columns]

In [6]:
time_range_info(basket.data)

for symbol, asset in basket.assets.items():
    mask = time_range_mask(asset.data, time_range['start_date'], time_range['end_date'])
    asset.data = asset.data[mask]

time_range_info(basket.data)

Data range: 2015-01-02 00:00:00 to 2025-12-22 00:00:00
Total duration: 4007 days 00:00:00
Data range: 2021-01-04 00:00:00 to 2024-12-31 00:00:00
Total duration: 1457 days 00:00:00


In [7]:
targets = ["Close"]
features = basket.get_unique_features()
print(f"Features:\t{features}\nTargets:\t{targets}")

Features:	['Close', 'High', 'Low', 'Open', 'Volume']
Targets:	['Close']


In [8]:
df = basket.data
df.ffill(inplace=True)
df.head()

AAPL                                                 \
                 Close        High         Low        Open     Volume   
Date                                                                    
2021-01-04  126.096588  130.189048  123.514437  130.101356  143301900   
2021-01-05  127.655609  128.366929  125.141666  125.589894   97664900   
2021-01-06  123.358521  127.694587  123.144152  124.449847  155088000   
2021-01-07  127.567940  128.259768  124.586290  125.073488  109578200   
2021-01-08  128.668976  129.234127  126.895568  129.039236  105158200   

                  TSLA                                                 ...  \
                 Close        High         Low        Open     Volume  ...   
Date                                                                   ...   
2021-01-04  243.256668  248.163330  239.063339  239.820007  145914600  ...   
2021-01-05  245.036667  246.946671  239.733337  241.220001   96735600  ...   
2021-01-06  251.993332  258.000000  249.699997  252.830002  134100000  ...   
2021-01-07  272.013336  272.329987  258.399994  259.209991  154496700  ...   
2021-01-08  293.339996  294.829987  279.463318  285.333344  225166500  ...   

                  AMD                                                  CSCO  \
                Close       High        Low       Open    Volume      Close   
Date                                                                          
2021-01-04  92.300003  96.059998  90.919998  92.110001  51802600  38.239326   
2021-01-05  92.769997  93.209999  91.410004  92.099998  34208000  38.256729   
2021-01-06  90.330002  92.279999  89.459999  91.620003  51911700  38.622074   
2021-01-07  95.160004  95.510002  91.199997  91.330002  42897200  39.109196   
2021-01-08  94.580002  96.400002  93.269997  95.980003  39816400  39.196186   

                                                       
                 High        Low       Open    Volume  
Date                                                   
2021-01-04  38.595972  37.708707  38.543782  24392500  
2021-01-05  38.335017  37.734811  37.995770  17763700  
2021-01-06  39.030909  38.178440  38.387210  21823100  
2021-01-07  39.239677  38.422000  38.448099  18218800  
2021-01-08  39.500638  38.491593  38.691662  20936300  

[5 rows x 70 columns]

In [9]:
assets = df.columns.get_level_values(0).unique()
assets

Index(['AAPL', 'TSLA', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'GOOG', 'META', 'AVGO',
       'ORCL', 'CRM', 'ADBE', 'AMD', 'CSCO'],
      dtype='object')

In [10]:
time_prd = window_size
processed_dfs = []

for symbol in assets:
    asset_df = df.xs(symbol, level=0, axis=1).copy()
    for target in targets:
        s = asset_df[target]

        # Calc Log Return 
        asset_df[f"Log_Returns {target}"] = np.log(s).diff()
        
        asset_df[f"SMA_{time_prd} {target}"] = ta.SMA(s, timeperiod=time_prd)
        asset_df[f"EMA_{time_prd} {target}"] = ta.EMA(s, timeperiod=time_prd)
        asset_df[f"RSI_{time_prd} {target}"] = ta.RSI(s, timeperiod=time_prd)
    asset_df.columns = pd.MultiIndex.from_product([[symbol], asset_df.columns])
    processed_dfs.append(asset_df)
    
df_updated = pd.concat(processed_dfs, axis=1)
df_updated.head()

AAPL                                                 \
                 Close        High         Low        Open     Volume   
Date                                                                    
2021-01-04  126.096588  130.189048  123.514437  130.101356  143301900   
2021-01-05  127.655609  128.366929  125.141666  125.589894   97664900   
2021-01-06  123.358521  127.694587  123.144152  124.449847  155088000   
2021-01-07  127.567940  128.259768  124.586290  125.073488  109578200   
2021-01-08  128.668976  129.234127  126.895568  129.039236  105158200   

                                                                     \
           Log_Returns Close SMA_64 Close EMA_64 Close RSI_64 Close   
Date                                                                  
2021-01-04               NaN          NaN          NaN          NaN   
2021-01-05          0.012288          NaN          NaN          NaN   
2021-01-06         -0.034241          NaN          NaN          NaN   
2021-01-07          0.033554          NaN          NaN          NaN   
2021-01-08          0.008594          NaN          NaN          NaN   

                  TSLA  ...          AMD       CSCO                        \
                 Close  ... RSI_64 Close      Close       High        Low   
Date                    ...                                                 
2021-01-04  243.256668  ...          NaN  38.239326  38.595972  37.708707   
2021-01-05  245.036667  ...          NaN  38.256729  38.335017  37.734811   
2021-01-06  251.993332  ...          NaN  38.622074  39.030909  38.178440   
2021-01-07  272.013336  ...          NaN  39.109196  39.239677  38.422000   
2021-01-08  293.339996  ...          NaN  39.196186  39.500638  38.491593   

                                                                             \
                 Open    Volume Log_Returns Close SMA_64 Close EMA_64 Close   
Date                                                                          
2021-01-04  38.543782  24392500               NaN          NaN          NaN   
2021-01-05  37.995770  17763700          0.000455          NaN          NaN   
2021-01-06  38.387210  21823100          0.009505          NaN          NaN   
2021-01-07  38.448099  18218800          0.012534          NaN          NaN   
2021-01-08  38.691662  20936300          0.002222          NaN          NaN   

                         
           RSI_64 Close  
Date                     
2021-01-04          NaN  
2021-01-05          NaN  
2021-01-06          NaN  
2021-01-07          NaN  
2021-01-08          NaN  

[5 rows x 126 columns]

In [11]:
nan_counts = df_updated.isna().sum()
print(nan_counts[nan_counts > 0])

AAPL   Log_Returns Close     1
       SMA_64 Close         63
       EMA_64 Close         63
       RSI_64 Close         64
TSLA   Log_Returns Close     1
       SMA_64 Close         63
       EMA_64 Close         63
       RSI_64 Close         64
MSFT   Log_Returns Close     1
       SMA_64 Close         63
       EMA_64 Close         63
       RSI_64 Close         64
NVDA   Log_Returns Close     1
       SMA_64 Close         63
       EMA_64 Close         63
       RSI_64 Close         64
GOOGL  Log_Returns Close     1
       SMA_64 Close         63
       EMA_64 Close         63
       RSI_64 Close         64
AMZN   Log_Returns Close     1
       SMA_64 Close         63
       EMA_64 Close         63
       RSI_64 Close         64
GOOG   Log_Returns Close     1
       SMA_64 Close         63
       EMA_64 Close         63
       RSI_64 Close         64
META   Log_Returns Close     1
       SMA_64 Close         63
       EMA_64 Close         63
       RSI_64 Close         64
AVGO   L

In [12]:
df_updated.dropna(inplace=True)
nan_counts = df_updated.isna().sum()
print(nan_counts[nan_counts > 0])
df_updated

Series([], dtype: int64)


AAPL                                                 \
                 Close        High         Low        Open     Volume   
Date                                                                    
2021-04-07  124.811455  124.830969  122.118102  122.791442   83466700   
2021-04-08  127.212051  127.241326  125.416488  125.836097   88844600   
2021-04-09  129.788330  129.827358  126.343573  126.665606  106686700   
2021-04-12  128.070831  129.641954  127.475561  129.319921   91420000   
2021-04-13  131.183777  131.408234  128.744147  129.241841   91266500   
...                ...         ...         ...         ...        ...   
2024-12-24  257.286682  257.296626  254.386957  254.586262   23234700   
2024-12-26  258.103729  259.179926  256.718662  257.276679   27237100   
2024-12-27  254.685867  257.784882  252.164818  256.917934   42355300   
2024-12-30  251.307877  252.603281  249.863009  251.337769   35557500   
2024-12-31  249.534180  252.384064  248.547676  251.547039   39480700   

                                                                     \
           Log_Returns Close SMA_64 Close EMA_64 Close RSI_64 Close   
Date                                                                  
2021-04-07          0.013301   125.016631   125.029780    49.492246   
2021-04-08          0.019051   125.009700   125.096927    50.447156   
2021-04-09          0.020050   125.110166   125.241278    51.447906   
2021-04-12         -0.013321   125.118024   125.328341    50.753732   
2021-04-13          0.024016   125.157318   125.508508    51.947529   
...                      ...          ...          ...          ...   
2024-12-24          0.011413   233.120571   234.909176    62.111132   
2024-12-26          0.003171   233.632796   235.622854    62.344220   
2024-12-27         -0.013331   234.073730   236.209408    60.755883   
2024-12-30         -0.013352   234.457685   236.673977    59.240547   
2024-12-31         -0.007083   234.732897   237.069675    58.462761   

                  TSLA  ...          AMD       CSCO                        \
                 Close  ... RSI_64 Close      Close       High        Low   
Date                    ...                                                 
2021-04-07  223.656662  ...    45.597975  45.355835  45.679992  45.189377   
2021-04-08  227.933334  ...    46.146395  45.478485  45.557334  45.075479   
2021-04-09  225.673340  ...    45.905210  45.636189  45.688756  45.276987   
2021-04-12  233.993332  ...    44.241014  45.180611  45.697511  45.093002   
2021-04-13  254.106674  ...    45.020897  45.259464  45.452208  44.979112   
...                ...  ...          ...        ...        ...        ...   
2024-12-24  462.279999  ...    44.764680  58.345390  58.345390  57.321788   
2024-12-26  454.130005  ...    44.480080  58.472118  58.550109  57.906701   
2024-12-27  431.660004  ...    44.517955  58.111423  58.511116  57.653238   
2024-12-30  417.410004  ...    43.874788  57.701981  57.896953  56.941591   
2024-12-31  403.839996  ...    43.491798  57.711735  57.887210  57.292545   

                                                                             \
                 Open    Volume Log_Returns Close SMA_64 Close EMA_64 Close   
Date                                                                          
2021-04-07  45.566098  15783600         -0.005009    41.167244    41.188350   
2021-04-08  45.504767  15111300          0.002701    41.280084    41.320354   
2021-04-09  45.408403  13135800          0.003462    41.389680    41.453149   
2021-04-12  45.618662  16441700         -0.010033    41.484546    41.567840   
2021-04-13  45.241942  13353200          0.001744    41.579284    41.681429   
...               ...       ...               ...          ...          ...   
2024-12-24  57.321788   9922300          0.014643    55.246462    55.055127   
2024-12-26  58.121168   8524500          0.002170    55.364197    55.160265   
2024-12-27  58.072428  13021400         -0.006188   

In [13]:
drop_cols = ['Open', 'High', 'Low', 'Volume']

df_final = df_updated.drop(columns=drop_cols, level=1).copy()
df_final

AAPL                                              \
                 Close Log_Returns Close SMA_64 Close EMA_64 Close   
Date                                                                 
2021-04-07  124.811455          0.013301   125.016631   125.029780   
2021-04-08  127.212051          0.019051   125.009700   125.096927   
2021-04-09  129.788330          0.020050   125.110166   125.241278   
2021-04-12  128.070831         -0.013321   125.118024   125.328341   
2021-04-13  131.183777          0.024016   125.157318   125.508508   
...                ...               ...          ...          ...   
2024-12-24  257.286682          0.011413   233.120571   234.909176   
2024-12-26  258.103729          0.003171   233.632796   235.622854   
2024-12-27  254.685867         -0.013331   234.073730   236.209408   
2024-12-30  251.307877         -0.013352   234.457685   236.673977   
2024-12-31  249.534180         -0.007083   234.732897   237.069675   

                               TSLA                                 \
           RSI_64 Close       Close Log_Returns Close SMA_64 Close   
Date                                                                 
2021-04-07    49.492246  223.656662         -0.030312   249.635208   
2021-04-08    50.447156  227.933334          0.018941   249.367968   
2021-04-09    51.447906  225.673340         -0.009965   248.956718   
2021-04-12    50.753732  233.993332          0.036204   248.362656   
2021-04-13    51.947529  254.106674          0.082462   247.749635   
...                 ...         ...               ...          ...   
2024-12-24    62.111132  462.279999          0.070991   311.215938   
2024-12-26    62.344220  454.130005         -0.017787   314.295782   
2024-12-27    60.755883  431.660004         -0.050745   317.068282   
2024-12-30    59.240547  417.410004         -0.033569   319.520626   
2024-12-31    58.462761  403.839996         -0.033050   321.742657   

                                      ...         AMD                    \
           EMA_64 Close RSI_64 Close  ...       Close Log_Returns Close   
Date                                  ...                                 
2021-04-07   249.132695    48.095904  ...   82.199997          0.009289   
2021-04-08   248.480407    48.530373  ...   83.349998          0.013893   
2021-04-09   247.778651    48.313270  ...   82.760002         -0.007104   
2021-04-12   247.354487    49.163780  ...   78.580002         -0.051828   
2021-04-13   247.562247    51.138336  ...   80.190002          0.020282   
...                 ...          ...  ...         ...               ...   
2024-12-24   333.380197    64.472803  ...  126.290001          0.013472   
2024-12-26   337.095576    63.613608  ...  125.059998         -0.009787   
2024-12-27   340.005251    61.324663  ...  125.190002          0.001039   
2024-12-30   342.386936    59.935288  ...  122.440002         -0.022211   
2024-12-31   344.277799    58.649838  ...  120.790001         -0.013568   

                                                        CSCO  \
           SMA_64 Close EMA_64 Close RSI_64 Close      Close   
Date                                                           
2021-04-07    85.720156    85.764800    45.597975  45.355835   
2021-04-08    85.572969    85.690499    46.146395  45.478485   
2021-04-09    85.454687    85.600330    45.905210  45.636189   
2021-04-12    85.195625    85.384320    44.241014  45.180611   
2021-04-13    84.970781    85.224494    45.020897  45.259464   
...                 ...          ...          ...        ...   
2024-12-24   146.457032   140.714791    44.764680  58.345390   
2024-12-26   145.879532   140.233105    44.480080  58.472118   
2024-12-27   145.218595   139.770240    44.517955  58.111423   
2024-12-30   144.563751   139.237002    43.874788  57.701981   
2024-12-31   143.887344   138.669402    43.491798  57.711735   

                                                                     
           Log_Returns Close SMA_64 Cl

In [14]:
close_price_df = df_final.xs('Close', level=1, axis=1)
close_price_df

,AAPL,TSLA,MSFT,NVDA,GOOGL,AMZN,GOOG,META,AVGO,ORCL,CRM,ADBE,AMD,CSCO
Date,,,,,,,,,,,,,,
2021-04-07,124.811455,223.656662,240.791092,14.109493,111.111977,163.969498,111.719475,310.918152,43.825417,69.652908,218.491653,493.410004,82.199997,45.355835
2021-04-08,127.212051,227.933334,244.018982,14.282575,111.677689,164.964996,112.502129,310.848602,44.099739,70.790741,222.113541,499.839996,83.349998,45.478485
2021-04-09,129.788330,225.673340,246.524216,14.365375,112.682106,168.610001,113.517166,310.292511,44.064320,71.072868,228.872452,504.040009,82.760002,45.636189
2021-04-12,128.070831,233.993332,246.582031,15.172430,111.389374,168.969498,111.973236,309.378876,43.935333,71.580658,226.378693,506.029999,78.580002,45.180611
2021-04-13,131.183777,254.106674,249.067993,15.641796,111.876183,170.000000,112.592995,307.611206,44.052517,72.097862,229.584946,514.859985,80.190002,45.259464
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-24,257.286682,462.279999,436.929138,140.189468,195.344940,229.050003,196.932236,605.839600,237.988037,169.721893,342.748352,447.940002,126.290001,58.345390
2024-12-26,258.103729,454.130005,435.715790,139.899521,194.836945,227.050003,196.463745,601.453369,243.627945,169.989227,340.051605,450.160004,125.059998,58.472118
2024-12-27,254.685867,431.660004,428.177216,136.980164,192.007996,223.750000,193.413620,597.924561,240.043442,167.296021,336.797607,446.480011,125.190002,58.111423


In [15]:
df_final.columns.remove_unused_levels()

n_obs = len(df_final)
n_assets = len(df_final.columns.get_level_values(0).unique())
n_features = len(df_final.columns.get_level_values(1).unique())

print(n_obs, n_assets, n_features)

market = df_final.values.reshape(n_obs, n_assets, n_features)
print(type(market))
market.shape

# tensor_list = []
     #    for symbol in self.symbols:
     #        if symbol in self.assets:
     #            # Call Asset method
     #            asset_tensor = self.assets[symbol].to_tensor(features, device)
     #            tensor_list.append(asset_tensor)
        
     #    # Stack along dimension 1 (Dimension N)
     #    # Asset: [T, F] -> Stack dim=1 -> [T, A, F]
     #    basket_tensor = torch.stack(tensor_list, dim=1)

941 14 5
<class 'numpy.ndarray'>


(941, 14, 5)

In [16]:
df_final.columns.levels[1]

Index(['Close', 'EMA_64 Close', 'High', 'Log_Returns Close', 'Low', 'Open',
       'RSI_64 Close', 'SMA_64 Close', 'Volume'],
      dtype='object')

In [17]:
n_prices = 1
n_targets = 1
# market[0, 0, 0] # -> that close price 

print(f"Market shape:\t\t{market.shape}")
print(f"Close price:\t\t{market[:,:,0:1].shape}")
print(f"Features_Target:\t{market[:,:,1:n_targets + 1].shape}")
print(f"Features_Condition:\t{market[:,:,n_targets + 1:].shape}")

print(f"That's a close price:\t{market[0,0,0:1]}")
print(f"Feature Target values:\t{market[0, 0, 1:n_targets + 1]}")
print(f"Feature_Cond values:\t{market[0, 0, n_targets + 1:]}")

price = market[:, :, 0:1]
x     = market[:, :, 1: n_targets + 1]
cond  = market[:, :, n_targets + 1:]
date  = df_final.index.get_level_values(0).unique().to_numpy()

print(f"price shape: {price.shape}")
print(f"x shape: {x.shape}")
print(f"cond shape: {cond.shape}")
print(f"date shape: {date.shape}")

Market shape:		(941, 14, 5)
Close price:		(941, 14, 1)
Features_Target:	(941, 14, 1)
Features_Condition:	(941, 14, 3)
That's a close price:	[124.81145477]
Feature Target values:	[0.01330142]
Feature_Cond values:	[125.01663101 125.02978025  49.49224583]
price shape: (941, 14, 1)
x shape: (941, 14, 1)
cond shape: (941, 14, 3)
date shape: (941,)


In [18]:
ratios = [0.8, 0.1, 0.1]
total_count = len(market)
train_count = int(total_count * ratios[0])
val_count = int(total_count * ratios[1])
test_count = total_count - train_count - val_count

print(f"Ratios DS\nTrain:\t{train_count}\nVal:\t{val_count}\nTest:\t{test_count}\nTotal:\t{total_count}")

Ratios DS
Train:	752
Val:	94
Test:	95
Total:	941


In [19]:
all_dates = df_final.index.get_level_values(0).unique().to_numpy()
print(f"Dates shape: {all_dates.shape}")

Dates shape: (941,)


In [20]:
all_prices = market[:, :, 0]
# market = market[:, :, 1:]

print(f"Prices shape: {all_prices.shape}")
print(f"Market shape: {market.shape}")

Prices shape: (941, 14)
Market shape: (941, 14, 5)


In [21]:
end_val = train_count + val_count
train_part = {
    "x": x[:train_count],
    "cond": cond[:train_count],
    "date": date[:train_count],
    "price": price[:train_count],
}

val_part = {
    "x": x[train_count:end_val],
    "cond": cond[train_count:end_val],
    "date": date[train_count:end_val],
    "price": price[train_count:end_val],
}

test_part = {
    "x": x[end_val:],
    "cond": cond[end_val:],
    "date": date[end_val:],
    "price": price[end_val:],
}

print(f"Train X: {train_part['x'].shape}\nVal X: {val_part['x'].shape}\nTest X:{test_part['x'].shape}")
print(f"-" * 30)
print(f"Train Cond: {train_part['cond'].shape}\nVal Cond: {val_part['cond'].shape}\nTest Cond:{test_part['cond'].shape}")
print(f"-" * 30)
print(f"Train Price: {train_part['price'].shape}\nVal Price: {val_part['price'].shape}\nTest Price:{test_part['price'].shape}")
print(f"-" * 30)
print(f"Train Dates: {train_part['date'].shape}\nVal Dates: {val_part['date'].shape}\nTest Dates:{test_part['date'].shape}")

Train X: (752, 14, 1)
Val X: (94, 14, 1)
Test X:(95, 14, 1)
------------------------------
Train Cond: (752, 14, 3)
Val Cond: (94, 14, 3)
Test Cond:(95, 14, 3)
------------------------------
Train Price: (752, 14, 1)
Val Price: (94, 14, 1)
Test Price:(95, 14, 1)
------------------------------
Train Dates: (752,)
Val Dates: (94,)
Test Dates:(95,)


In [22]:
scaler_x = StandardScaler()
scaler_cond = StandardScaler()

# Scale X
x_train_2d = train_part['x'].reshape(train_part['x'].shape[0], -1)
x_val_2d = val_part['x'].reshape(val_part['x'].shape[0], -1)
x_test_2d = test_part['x'].reshape(test_part['x'].shape[0], -1)

scaler_x.fit(x_train_2d)

train_x_scaled = scaler_x.transform(x_train_2d).reshape(train_part['x'].shape)
val_x_scaled = scaler_x.transform(x_val_2d).reshape(val_part['x'].shape)
test_x_scaled = scaler_x.transform(x_test_2d).reshape(test_part['x'].shape)

inspect(train_x_scaled, "X_scaled - Train Part")
inspect(val_x_scaled, "X_scaled - Val Part")
inspect(test_x_scaled, "X_scaled - Test Part")

# Scale Cond
cond_train_2d = train_part['cond'].reshape(train_part['cond'].shape[0], -1)
cond_val_2d = val_part['cond'].reshape(val_part['cond'].shape[0], -1)
cond_test_2d = test_part['cond'].reshape(test_part['cond'].shape[0], -1)

scaler_cond.fit(cond_train_2d)

train_cond_scaled = scaler_cond.transform(cond_train_2d).reshape(train_part['cond'].shape)
val_cond_scaled = scaler_cond.transform(cond_val_2d).reshape(val_part['cond'].shape)
test_cond_scaled = scaler_cond.transform(cond_test_2d).reshape(test_part['cond'].shape)

inspect(train_cond_scaled, "Cond_scaled - Train Part")
inspect(val_cond_scaled, "Cond_scaled - Val Part")
inspect(test_cond_scaled, "Cond_scaled - Test Part")

--- Inspecting: X_scaled - Train Part ---
------------------------------------
Shape: (752, 14, 1)
Min:   -10.0955
Max:   7.6114
Mean:  -0.0000
Std:   1.0000
------------------------------------
--- Inspecting: X_scaled - Val Part ---
------------------------------------
Shape: (94, 14, 1)
Min:   -9.7290
Max:   6.5545
Mean:  0.0105
Std:   1.0389
------------------------------------
--- Inspecting: X_scaled - Test Part ---
------------------------------------
Shape: (95, 14, 1)
Min:   -6.2410
Max:   10.5866
Mean:  0.0425
Std:   0.9578
------------------------------------
--- Inspecting: Cond_scaled - Train Part ---
------------------------------------
Shape: (752, 14, 3)
Min:   -2.5162
Max:   4.2147
Mean:  -0.0000
Std:   1.0000
------------------------------------
--- Inspecting: Cond_scaled - Val Part ---
------------------------------------
Shape: (94, 14, 3)
Min:   -1.9789
Max:   6.8066
Mean:  1.4218
Std:   1.6786
------------------------------------
--- Inspecting: Cond_scaled - Tes

(np.float64(-1.5747055002662316),
 np.float64(8.408768777257071),
 np.float64(2.023259768035455),
 np.float64(2.0689639135157867))

In [23]:
# Pipeline([('Scaler_X', scaler_x), ('Scaler_Cond', scaler_cond)])

In [24]:
train_part['date'].shape
train_part['price'].shape

(752, 14, 1)

In [25]:
train_ds = MarketDataset(x=train_x_scaled, cond=train_cond_scaled, date=train_part['date'], price=train_part['price'], window_size=window_size, stride=stride)
val_ds = MarketDataset(x=val_x_scaled, cond=val_cond_scaled, date=val_part['date'], price=val_part['price'], window_size=window_size, stride=stride)
test_ds = MarketDataset(x=test_x_scaled, cond=test_cond_scaled, date=test_part['date'], price=test_part['price'], window_size=window_size, stride=stride)

print(f"Num of Windows\nTrain DS: {len(train_ds)}, Val Ds: {len(val_ds)}, Test DS: {len(test_ds)}\n")
print(f"A sample shape from Train DS\n\tx: {train_ds[0]['x'].shape},\n\tcond: {train_ds[0]['cond'].shape}\n\tdates: {len(train_ds[0]['date'])}")

Num of Windows
Train DS: 138, Val Ds: 7, Test DS: 7

A sample shape from Train DS
	x: (1, 64, 14),
	cond: (3, 64, 14)
	dates: 64


In [26]:
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

batch_train = next(iter(train_loader))
print(len(train_loader))
print(batch_train["x"].shape)
print(batch_train["cond"].shape)

5
torch.Size([32, 1, 64, 14])
torch.Size([32, 3, 64, 14])


In [27]:
len(test_loader)

1

In [28]:
next(iter(test_loader))["x"].shape

torch.Size([7, 1, 64, 14])

In [29]:
B, C_cond, T, A = batch_train["cond"].shape
B, C_target, T, A = batch_train["x"].shape
print(f"B: {B}, C_target: {C_target}, C_cond: {C_cond}, T: {T}, A: {A}")

B: 32, C_target: 1, C_cond: 3, T: 64, A: 14


In [30]:
def objective(trial):
    d_model = trial.suggest_categorical("d_model", [64, 128, 256])

    valid_heads = [h for h in [2, 4, 8] if d_model % h == 0]
    nhead = trial.suggest_categorical("nhead", valid_heads)

    num_layers = trial.suggest_int("num_layers", 2, 8)
    dim_feedforward = trial.suggest_int("dim_feedforward", 256, 1024, step=128)

    # 2. Optimization Params
    # lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    # weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)

    # timesteps = trial.suggest_categorical("timesteps", [500, 1000])
    # beta_start = trial.suggest_float("beta_start", 1e-5, 1e-3, log=True)
    # beta_end = trial.suggest_float("beta_end", 0.01, 0.05)

    # if beta_start >= beta_end:
        # beta_start, beta_end = beta_end, beta_start
    
    transformer = DiffusionTransformer(
        num_assets          = A,
        num_channels        = C_target,
        num_cond_channels   = C_cond,
        num_layers          = num_layers,
        num_attention_heads = nhead,
        seq_length          = T,
        d_model             = d_model,
        dim_feedforward     = dim_feedforward,
        dropout             = ddpm_transformer['dropout']
    ).to(device)

    diffusion_model = Diffusion(
        model=transformer,
        timesteps=ddpm['timesteps'],
        beta_start=ddpm['beta_start'],
        beta_end=ddpm['beta_end']
    ).to(device)

    optimizer = optim.AdamW(diffusion_model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )

    checkpoint_filename = f"tr{trial.number}_d{d_model}_l{num_layers}_h{nhead}_t{ddpm['timesteps']}.pt"


    engine = Engine(
        train_loader    = train_loader,
        val_loader      = val_loader,
        model           = diffusion_model,
        optimizer       = optimizer,
        criterion       = nn.MSELoss(),
        scheduler       = scheduler,
        device          = device,
        checkpoint_dir  = os.path.join(RESULTS_DIR,"optuna_checkpoints"),
        checkpoint_filename = checkpoint_filename
    )


    EPOCHS_PER_TRIAL = 50

    for epoch in range(1, EPOCHS_PER_TRIAL + 1):
        try:
            # Train & Validate
            train_loss = engine.train(epoch)
            val_loss = engine.validate(epoch)
            
            # --- Key Step: report result to Optuna ---
            trial.report(val_loss, epoch)

            # --- Key Step: cut if it's bad (Pruning) ---
            if trial.should_prune():
                print(f"Trial {trial.number} pruned at epoch {epoch}")
                raise optuna.exceptions.TrialPruned()
                
            # Update Scheduler
            scheduler.step(val_loss)

        except RuntimeError as e:
            # if OOM (Memory is full) skip this Trial  no Crash
            if "out of memory" in str(e):
                torch.cuda.empty_cache()
                raise optuna.exceptions.TrialPruned()
            else:
                raise e

    return val_loss

In [31]:
model = DiffusionTransformer(
    num_assets          = A,
    num_channels        = C_target,
    num_cond_channels   = C_cond,
    num_layers          = ddpm_transformer['num_layers'],
    num_attention_heads = ddpm_transformer['nhead'],
    seq_length          = T,
    d_model             = ddpm_transformer['d_model'],
    dim_feedforward     = ddpm_transformer['dim_feedforward'],
    dropout             = ddpm_transformer['dropout']
).to(device)

diffusion = Diffusion(
    model=model,
    timesteps=ddpm['timesteps'],
    beta_start=ddpm['beta_start'],
    beta_end=ddpm['beta_end']
).to(device)

optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, cooldown=2, threshold=0.01)

engine = Engine(
    train_loader    = train_loader,
    val_loader      = val_loader,
    model           = diffusion,
    optimizer       = optimizer,
    criterion       = nn.MSELoss(),
    scheduler       = scheduler,
    device          = device
)

input dim x: 14, d_model: 64


2026-02-11 17:49:08,834 - Engine - INFO - Engine initialized on cuda:1
2026-02-11 17:49:08,836 - Engine - INFO - Criterion: MSELoss


In [32]:
diffusion = Diffusion(
    model=model,
    timesteps=ddpm['timesteps'],
    beta_start=ddpm['beta_start'],
    beta_end=ddpm['beta_end']
).to(device)

In [33]:
optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

In [34]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, cooldown=2, threshold=0.01)

In [35]:
engine = Engine(
    train_loader    = train_loader,
    val_loader      = val_loader,
    model           = diffusion,
    optimizer       = optimizer,
    criterion       = nn.MSELoss(),
    scheduler       = scheduler,
    device          = device
)

2026-02-11 17:49:09,241 - Engine - INFO - Engine initialized on cuda:1
2026-02-11 17:49:09,242 - Engine - INFO - Criterion: MSELoss


In [36]:
# engine.fit(epochs)
engine.fit(1000)

2026-02-11 17:49:09,410 - Engine - INFO - Starting training for 1000 epochs...
Train Ep 1: 100%|██████████| 5/5 [00:02<00:00,  2.49it/s, loss=0.9791]
2026-02-11 17:49:11,500 - Engine - INFO - Epoch 1 | Val Loss: 0.9790
2026-02-11 17:49:11,894 - Engine - INFO - New best model saved! (Val Loss: 0.9790)
Train Ep 2: 100%|██████████| 5/5 [00:01<00:00,  3.49it/s, loss=0.9824]
2026-02-11 17:49:13,433 - Engine - INFO - Epoch 2 | Val Loss: 0.9940
Train Ep 3: 100%|██████████| 5/5 [00:01<00:00,  3.03it/s, loss=0.9350]
2026-02-11 17:49:15,180 - Engine - INFO - Epoch 3 | Val Loss: 0.9640
2026-02-11 17:49:15,612 - Engine - INFO - New best model saved! (Val Loss: 0.9640)
Train Ep 4: 100%|██████████| 5/5 [00:01<00:00,  3.26it/s, loss=0.9327]
2026-02-11 17:49:17,231 - Engine - INFO - Epoch 4 | Val Loss: 1.0121
Train Ep 5: 100%|██████████| 5/5 [00:01<00:00,  3.69it/s, loss=0.8663]
2026-02-11 17:49:18,657 - Engine - INFO - Epoch 5 | Val Loss: 0.9736
Train Ep 6: 100%|██████████| 5/5 [00:01<00:00,  4.30it/

In [ ]:
import optuna
from optuna.trial import TrialState
import os


n_trails = 50
n_warmup = 10

study = optuna.create_study(
        study_name="diffusion_tuning",
        direction="minimize",  # เราต้องการ Loss ต่ำสุด
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=n_warmup) # ให้โอกาส 3 Epoch แรกก่อนค่อยตัด
    )

print("🚀 Starting Optuna Tuning...")


study.optimize(objective, n_trials=n_trails)

print("\n==================================")
print("✅ Tuning Complete!")
print("==================================")
print(f"Best Loss: {study.best_value:.6f}")
print("Best Params:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

best_params = study.best_params

In [ ]:
from optuna.visualization import plot_param_importances, plot_optimization_history

# 1. ดูว่า Param ไหนส่งผลต่อ Loss มากสุด
fig1 = plot_param_importances(study)
fig1.show(renderer="notebook")

# 2. ดูแนวโน้มว่า Loss ลดลงเรื่อยๆ ไหม
fig2 = plot_optimization_history(study)
fig2.show(renderer="notebook")

In [ ]:
study

In [ ]:
import optuna.visualization as vis

fig1 = vis.plot_optimization_history(study)
fig1.show(renderer="notebook")

In [ ]:
best_params = study.best_params
best_params

In [ ]:
print(f"lr: {lr}, weight_decay: {weight_decay},  ddpm_transformer['dropout']: { ddpm_transformer['dropout']}")

In [ ]:
# ignore droupout
# lr 
# too high

model = DiffusionTransformer(
    num_assets          = A,
    num_channels        = C_target,
    num_cond_channels   = C_cond,
    num_layers          = best_params['num_layers'],
    num_attention_heads = best_params['nhead'],
    seq_length          = T,
    d_model             = best_params['d_model'],
    dim_feedforward     = best_params['dim_feedforward'],
    dropout             = ddpm_transformer['dropout']
).to(device)

diffusion = Diffusion(
    model=model,
    timesteps=ddpm['timesteps'],
    beta_start=ddpm['beta_start'],
    beta_end=ddpm['beta_end']
).to(device)

optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, cooldown=2, threshold=0.01)



checkpoint_filename = f"tr{n_trails}_{n_warmup}_d{best_params['d_model']}_l{best_params['num_layers']}_h{best_params['nhead']}_t{ddpm['timesteps']}.pt"

engine = Engine(
    train_loader    = train_loader,
    val_loader      = val_loader,
    model           = diffusion,
    optimizer       = optimizer,
    criterion       = nn.MSELoss(),
    scheduler       = scheduler,
    device          = device,
    checkpoint_dir  = os.path.join(RESULTS_DIR,"optuna_checkpoints"),
    checkpoint_filename = checkpoint_filename
)


epochs = 1000

In [ ]:
engine.fit(epochs)

In [ ]:
12